In [ ]:
import gensim
import torch

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import numpy as np

In [ ]:
!pip install annoy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.5/647.5 kB 4.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for annoy: filename=annoy-1.17.3-cp310-cp310-linux_x86_64.whl size=552399 sha256=9db3bae2b638379e4995632172a92c20f1e7eb999f53c58b873312a50f241ef2
  Stored in directory: /root/.cache/pip/wheels/64/8a/da/f714bcf46c5efdcfcac0559e63370c21abe961c48e3992465a
Successfully built annoy


In [ ]:
from numpy.random import normal
from annoy import AnnoyIndex

In [ ]:
from gensim.scripts.glove2word2vec import glove2word2vec

In [ ]:
from gensim.models import KeyedVectors

# Path to the original GloVe embeddings file
glove_file_path = '/content/drive/MyDrive/glove.6B.50d.txt'

processed_file_path = '/content/Preprocessed.glove'

glove2word2vec(glove_file_path,processed_file_path)

<ipython-input-9-0e5e1ed340fd>:8: DeprecationWarning: Call to deprecated `glove2word2vec` (KeyedVectors.load_word2vec_format(.., binary=False, no_header=True) loads GLoVE text vectors.).
  glove2word2vec(glove_file_path,processed_file_path)


(400001, 50)

In [ ]:
model = KeyedVectors.load_word2vec_format(processed_file_path,binary = False)

In [ ]:
model['word']

array([-0.1643   ,  0.15722  , -0.55021  , -0.3303   ,  0.66463  ,
       -0.1152   , -0.2261   , -0.23674  , -0.86119  ,  0.24319  ,
        0.074499 ,  0.61081  ,  0.73683  , -0.35224  ,  0.61346  ,
        0.0050975, -0.62538  , -0.0050458,  0.18392  , -0.12214  ,
       -0.65973  , -0.30673  ,  0.35038  ,  0.75805  ,  1.0183   ,
       -1.7424   , -1.4277   ,  0.38032  ,  0.37713  , -0.74941  ,
        2.9401   , -0.8097   , -0.66901  ,  0.23123  , -0.073194 ,
       -0.13624  ,  0.24424  , -1.0129   , -0.24919  , -0.06893  ,
        0.70231  , -0.022177 , -0.64684  ,  0.59599  ,  0.027092 ,
        0.11203  ,  0.61214  ,  0.74339  ,  0.23572  , -0.1369   ],
      dtype=float32)

In [ ]:
import random

def get_random_nearest_neighbour(model, noisy_vector):
    # Find the top 5 nearest neighbors to the noisy vector
    nearest_neighbors = model.most_similar(positive=[noisy_vector], topn=5)

    # Generate a random number between 1 and 5 (inclusive)
    random_number = random.randint(1, 5)

    # Use the random number to select a nearest neighbor
    random_nearest_neighbor = nearest_neighbors[random_number - 1]


    return random_nearest_neighbor

In [ ]:
def generate_laplacian_noise_vector(dimension, sensitivity, epsilon):
    rand_vec = normal(size=dimension)
    normalized_vec = rand_vec / np.linalg.norm(rand_vec)
    magnitude = np.random.gamma(shape=dimension, scale=sensitivity / epsilon)
    return normalized_vec * magnitude

In [ ]:
def replace_word(sensitive_word):

    # Generate a noise vector
    noise = generate_laplacian_noise_vector(50, 1.75, 25)
    # Get vector of sensitive word
    original_vec = model[sensitive_word]
    # Get perturbed vector
    noisy_vector = original_vec + noise
    # Get item closest to noisy vector
    privatized_word = get_random_nearest_neighbour(model,noisy_vector)[0]
    # Get word from item

    return privatized_word

In [ ]:
def replace_sentence(sentence, model):
    # Split the sentence into words
    words = sentence.split()
    privatized_sentence = []

    for word in words:
        try:
            # Generate a noise vector
            noise = generate_laplacian_noise_vector(50, 1.5, 25)
            # Get vector of sensitive word
            original_vec = model[word]
            # Get perturbed vector
            noisy_vector = original_vec + noise
            # Get item closest to noisy vector
            privatized_word = get_random_nearest_neighbour(model, noisy_vector)[0]
            # Add the privatized word to the privatized sentence
            privatized_sentence.append(privatized_word)
        except KeyError:
            # If the word is not in the vocabulary, keep it as is
            privatized_sentence.append(word)

    # Join the privatized words to form the privatized sentence
    return ' '.join(privatized_sentence)

In [ ]:
import pandas as pd


df = pd.read_csv('/content/drive/MyDrive/IMDB_Dataset.csv')
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [ ]:
import numpy as np
import re
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize

In [ ]:
# Download required NLTK datasets
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


True

In [ ]:
# Define the list of stopwords and WordNetLemmatizer
stopwords = nltk.corpus.stopwords.words('english')
wn = nltk.WordNetLemmatizer()

In [ ]:
# Define a function to clean the text
def clean_text(text):
    # Remove non-alphabetic characters
    text = re.sub('[^a-zA-Z]', ' ', text)
    # Convert to lowercase
    text = text.lower()
    # Tokenize the text
    text = nltk.word_tokenize(text)
    # Lemmatize the words and remove stopwords
    text = [wn.lemmatize(word) for word in text if word not in stopwords]
    # Join the words back into a string
    text = ' '.join(text)
    return text

In [ ]:
df['review'] = df['review'].apply(clean_text)

In [ ]:
df.head()


,review,sentiment
0,one reviewer mentioned watching oz episode hoo...,positive
1,wonderful little production br br filming tech...,positive
2,thought wonderful way spend time hot summer we...,positive
3,basically family little boy jake think zombie ...,negative
4,petter mattei love time money visually stunnin...,positive


In [ ]:
embeddings = torch.tensor(model.vectors)


# Move the PyTorch tensor to the GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
embeddings = embeddings.to(device)

In [ ]:
batch_size = 100
total_rows = len(df)
for i in range(0, total_rows, batch_size):
    subset_df = df.iloc[i:i+batch_size]
    subset_df['privatized_text'] = subset_df['review'].apply(lambda x: replace_sentence(x, model))
    subset_df.to_csv("/content/privatized_subset_imdb_dataset_{i+1}_{i+batch_size}.csv", index=False)
    print(f"Processed rows {i+1} to {min(i+batch_size, total_rows)}")
    print(subset_df['privatized_text'])

<ipython-input-30-5d1840b45205>:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  subset_df['privatized_text'] = subset_df['review'].apply(lambda x: replace_sentence(x, model))


Processed rows 1 to 100
0     only allmusic mention talk trilogy simpsons su...
1     reminds much produce db bb live-action methods...
2     never something view bringing starting spots a...
3     so lived lot boy trevor ca zombies sofa acquir...
4     solberg mattei ` when cash visual spectacular ...
                            ...                        
95    martin week stewart dependable actor dying wel...
96    maybe could originally sure than three part co...
97    and most wait too horror b movies cause we fas...
98    worst movies because seen made past probably e...
99    bruno as long wonderful remembered legacy play...
Name: privatized_text, Length: 100, dtype: object


<ipython-input-30-5d1840b45205>:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  subset_df['privatized_text'] = subset_df['review'].apply(lambda x: replace_sentence(x, model))


Processed rows 101 to 200
100    short film inspired would short height feature...
101    okay next night june th cultures exasperation ...
102    bought games perverse sell wal sorry memorable...
103    amusingly awful made movies films shinai blow ...
104    hand afraid early comedy just happening die ma...
                             ...                        
195    mfecane classes hizzoner ii fantastic kurigalz...
196    ludicrous nystagmus last man vandermark instea...
197    candace grant joyce take go legend “ b movie k...
198    put flexible historical sense film though seem...
199    directed coming sure including lynch scenes so...
Name: privatized_text, Length: 100, dtype: object


<ipython-input-30-5d1840b45205>:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  subset_df['privatized_text'] = subset_df['review'].apply(lambda x: replace_sentence(x, model))


...

Processed rows 5001 to 5100
5000    films turn ` four friend black starring scenes...
5001    movies type movies pricey being getting bad of...
5002    polygram broadcast box made helped championshi...
5003    bukovina statistically occasionally funny quir...
5004    exactly rated acts comedy noticeably history m...
                              ...                        
5095    comedy weird watched poisoning drop psilcybe l...
5096    goldie hawn sir buyer films highest only wo su...
5097    caddyshack similarly ambient spooky funny orig...
5098    rents couple movies much granddaughter this la...
5099    brought movies comic main finally movies somet...
Name: privatized_text, Length: 100, dtype: object


<ipython-input-30-5d1840b45205>:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  subset_df['privatized_text'] = subset_df['review'].apply(lambda x: replace_sentence(x, model))


Processed rows 5101 to 5200
5100    quality bros. elektra meant starred milam anth...
5101    seen movies gorgeous flying onecare cinema han...
5102    film portrays rented film intensity horrible c...
5103    documentary proposal playful only near good co...
5104    scintillating fiction small group idealistic m...
                              ...                        
5195    really horror cheer very dreams porky often sa...
5196    game cutscenes film truthfulness waited one ch...
5197    well movie horrible socialize new screens star...
5198    official hour bleeds awful than as dismal mora...
5199    glencairn picture also movie g f such army uni...
Name: privatized_text, Length: 100, dtype: object


<ipython-input-30-5d1840b45205>:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  subset_df['privatized_text'] = subset_df['review'].apply(lambda x: replace_sentence(x, model))


Processed rows 5201 to 5300
5200    conflict iraq killed joining soldier flesh inh...
5201    two hollywood especially 've cherish second in...
5202    obviously majestic heartwrenching films ripste...
5203    greatest role ever movie things built do movie...
5204    crazy theme longtime hard-drinking player coin...
                              ...                        
5295    might perhaps hoping terminator sports thought...
5296    astronaut bob north davey multinucleated comra...
5297    well gone noting only aftermath adaptation bes...
5298    spoiler switchable ramped by put suspense rele...
5299    each believe film thankfully without get disho...
Name: privatized_text, Length: 100, dtype: object
